In [1]:
import torch 
import torch.nn as nn

In [ ]:
class NoisyTopKGating(nn.Module):
    def __init__(self, D, N, K):
        super(NoisyTopKGating, self).__init__()
        self.W_G = torch.nn.Parameter(torch.randn((D, N)))
        self.W_N = torch.nn.Parameter(torch.randn((D, N)))
        self.normal_dist = torch.distributions.Normal(loc=0, scale=1)
        self.softplus = nn.Softplus()

        self.D = D
        self.N = N 
        self.K = K


    def compute_moe_aux_loss(self, H, KV, KI):
        (B, S, _) = H.shape
        mask = torch.ones_like(H, dtype=torch.bool)
        mask.scatter_(dim=-1, index=KI, value=False)

        H = H.masked_fill(mask, float("-inf"))


        assert KI.shape == (B, S, self.K)


        G = nn.Softmax(dim=-1)(H)

        assert G.shape == (B, S, self.N)

        probs = G.mean(dim=(0, 1))

        assert probs.shape == (self.N, )

        threshold_logit = KV[:,:,-1:]

        assert threshold_logit.shape == (B, S, 1)

        f = torch.randn((self.N, )) # to do later

     

        aux_loss = torch.sum(f * probs)

        return G, aux_loss


    def forward(self, X:torch.tensor): # (B, S, D)
        (B, S, D) = X.shape
        W_G = X @ self.W_G # (B, S, D) @ (D, N) = (B, S, N)
        W_N = X @ self.W_N

        assert W_G.shape == (B, S, self.N)
        
        e = self.normal_dist.sample((B, S, self.N)) 

        H = W_G + e * self.softplus(W_N) # (B, S, N)
        KV, KI = torch.topk(H, k=self.K, dim=-1) # (B, S, K)

        return self.compute_moe_aux_loss(H, KV, KI), KI        



        



noisy_top_k_gating = NoisyTopKGating(D=2, N=5, K=3)

X = torch.randn((1, 4, 2)).float()
noisy_top_k_gating.forward(X)

((tensor([[[0.0074, 0.9908, 0.0000, 0.0019, 0.0000],
           [0.2639, 0.4254, 0.0000, 0.3107, 0.0000],
           [0.1288, 0.0000, 0.7684, 0.1027, 0.0000],
           [0.0000, 0.6764, 0.1574, 0.1662, 0.0000]]],
         grad_fn=<SoftmaxBackward0>),
  tensor(-0.3546, grad_fn=<SumBackward0>)),
 tensor([[[1, 0, 3],
          [1, 3, 0],
          [2, 0, 3],
          [1, 3, 2]]]))

In [3]:
a = torch.randn((4, 3))
print(a)

b = torch.tensor([
    [1, 2],
    [2, 0],
    [0, 1],
    [1, 0]
    ])
torch.gather(a, dim=1, index=b)

tensor([[ 0.8270,  0.7845,  0.8444],
        [ 0.6714, -1.5194,  0.7141],
        [ 0.0565, -0.4625,  0.3808],
        [ 0.5799,  1.8144, -0.1159]])


tensor([[ 0.7845,  0.8444],
        [ 0.7141,  0.6714],
        [ 0.0565, -0.4625],
        [ 1.8144,  0.5799]])